# 02 — Elevation data

Downloads and mosaics the 1 m DHMV II surface and terrain models required by the analysis.


In [ ]:
# Load the acquisition modules and define the raw, interim, and external data directories.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from eclipse_viewshed.project import repository_root
PROJECT_ROOT = repository_root()

from eclipse_viewshed import acquire, aoi, solar

RAW = PROJECT_ROOT / "data" / "raw" / "dhmv_tiles"
INTERIM = PROJECT_ROOT / "data" / "interim"
EXTERNAL = PROJECT_ROOT / "data" / "external"
FIGURES = PROJECT_ROOT / "reports" / "figures"
for directory in (RAW, INTERIM, FIGURES):
    directory.mkdir(parents=True, exist_ok=True)


def window_around(src, x, y, half_m):
    """Return a raster window clipped to the dataset extent."""
    bounds = src.bounds
    x0, x1 = max(x - half_m, bounds.left), min(x + half_m, bounds.right)
    y0, y1 = max(y - half_m, bounds.bottom), min(y + half_m, bounds.top)
    window = rasterio.windows.from_bounds(
        x0, y0, x1, y1, transform=src.transform
    )
    return window, (x0, y0, x1, y1)


plt.rcParams["figure.dpi"] = 110


## Observation points


In [ ]:
# Load the observer table and project every coordinate into Lambert 72.
observers = pd.read_csv(EXTERNAL / "observers.csv")

# Convert the observer coordinates to the metric analysis CRS.
observers[["x_l72", "y_l72"]] = observers.apply(
    lambda r: pd.Series(aoi.to_lambert72(r["lat"], r["lon"])), axis=1)

observers


## Required coverage


In [ ]:
# Calculate the analysis and map bounds and enumerate the required elevation tiles.
primary = observers.iloc[0]
px, py = primary["x_l72"], primary["y_l72"]

fan = aoi.fan_bounds(px, py, ray_m=aoi.DEFAULT_RAY_M)
tiles_primary = aoi.tiles_covering(fan)

# Cover every observer's sightline fan.
points = list(zip(observers["lat"], observers["lon"]))
tiles_fans = aoi.tiles_for_observers(points, ray_m=aoi.DEFAULT_RAY_M)

# Add the area required by the site map. Union the two so the map frame is covered.
tiles_map = aoi.tiles_for_map(points)
tiles_needed = sorted(set(tiles_fans) | set(tiles_map))

print(f"{primary['name']}  (primary)")
print(f"  Lambert 72 : X={px:.0f}  Y={py:.0f}")
print(f"  fan bounds : {fan}")
print(f"  area       : {fan.area_km2:.0f} km2")
print(f"  tiles      : {len(tiles_primary)}")
print()
print(f"all {len(observers)} observers")
print(f"  fan tiles  : {len(tiles_fans)}")
print(f"  map frame  : {aoi.map_frame(points)}")
print(f"  map tiles  : {len(tiles_map)}  "
      f"({len(set(tiles_map) - set(tiles_fans))} the fans never asked for)")
print(f"  union      : {len(tiles_needed)} tiles")
print(f"  ~download  : {len(tiles_needed) * aoi.TILE_M**2 * 4 / 1e6:.0f} MB "
      f"uncompressed, x2 for DSM+DTM")


## Service metadata


In [ ]:
# Read the DSM and DTM service metadata and verify their shared grid definition.
dsm_info = acquire.describe_coverage(acquire.COVERAGE_DSM)
dtm_info = acquire.describe_coverage(acquire.COVERAGE_DTM)

for info in (dsm_info, dtm_info):
    print(f"{info.coverage_id}")
    print(f"  CRS         : {info.crs}")
    print(f"  axis labels : {info.axis_labels}")
    print(f"  pixel size  : {info.pixel_size[0]} x {info.pixel_size[1]} m")
    print(f"  nodata      : {info.nodata}")
    print(f"  extent      : {info.envelope}")
    print(f"  format      : {info.native_format}\n")

# Validate the grid assumptions used by the analysis.
assert dsm_info.crs == "EPSG:31370", f"unexpected CRS {dsm_info.crs}"
assert dsm_info.pixel_size == (1.0, 1.0), f"not a 1 m grid: {dsm_info.pixel_size}"
assert dsm_info.axis_labels == dtm_info.axis_labels, "DSM and DTM axes differ"
print("Service metadata matches expectations.")


## Tile acquisition


In [ ]:
# Download any missing DSM and DTM tiles and record their provenance.
report_dsm = acquire.ensure_tiles(
    acquire.COVERAGE_DSM, tiles_needed, RAW, axis_labels=axis)
print("DSM:", report_dsm)

report_dtm = acquire.ensure_tiles(
    acquire.COVERAGE_DTM, tiles_needed, RAW, axis_labels=axis)
print("DTM:", report_dtm)

assert not report_dsm.failed and not report_dtm.failed, "One or more elevation tiles failed"


## Mosaics


In [ ]:
# Build virtual mosaics from the downloaded DSM and DTM tiles.
suffix = f"_t{aoi.TILE_M}.tif"
dsm_tiles = sorted(RAW.glob(f"{acquire.COVERAGE_DSM}_*{suffix}"))
dtm_tiles = sorted(RAW.glob(f"{acquire.COVERAGE_DTM}_*{suffix}"))
print(f"mosaicking {len(dsm_tiles)} DSM and {len(dtm_tiles)} DTM tiles "
      f"at {aoi.TILE_M} m")

dsm_vrt = acquire.build_vrt(dsm_tiles, INTERIM / "dsm_1m.vrt")
dtm_vrt = acquire.build_vrt(dtm_tiles, INTERIM / "dtm_1m.vrt")

for name, path in (("DSM", dsm_vrt), ("DTM", dtm_vrt)):
    with rasterio.open(path) as src:
        print(f"{name}: {src.width} x {src.height} px, {src.crs}, "
              f"res {src.res}, {src.width*src.height/1e6:.0f} Mpx")


## Validation


In [ ]:
# Validate the mosaics' alignment, coverage, vertical datum, and surface ordering.
checks = {}
notes = {}

with rasterio.open(dsm_vrt) as dsm, rasterio.open(dtm_vrt) as dtm:

    # Confirm that the surface and terrain rasters share one grid.
    checks["DSM and DTM grids align"] = (
        dsm.transform == dtm.transform
        and (dsm.width, dsm.height) == (dtm.width, dtm.height))

    # Confirm that the observer falls inside the mosaic.
    b = dsm.bounds
    checks["primary observer inside extent"] = (
        b.left <= px <= b.right and b.bottom <= py <= b.top)

    # Sample a two-kilometre box around the observer.
    win, wb = window_around(dsm, px, py, 1000)
    dsm_s = dsm.read(1, window=win, masked=True)
    dtm_s = dtm.read(1, window=win, masked=True)
    box_km = ((wb[2]-wb[0])/1000, (wb[3]-wb[1])/1000)
    ndsm = (dsm_s - dtm_s).compressed()

    # Check that the terrain values are plausible metres TAW.
    ground = float(np.ma.median(dtm_s))
    checks["ground level consistent with TAW"] = -2.0 < ground < 25.0

    # Confirm that the surface raster rises above the terrain raster.
    p99 = float(np.percentile(ndsm, 99))
    checks["DSM sits above DTM (not swapped)"] = p99 > 5.0

    # Record compact diagnostics for review.
    notes["bare-earth median"] = f"{ground:.2f} m TAW"
    notes["DSM range in box"] = f"{dsm_s.min():.2f} to {dsm_s.max():.2f} m TAW"
    notes["object height p99"] = f"{p99:.2f} m"
    notes["object height p1"] = f"{np.percentile(ndsm, 1):.2f} m"
    notes["pixels with DSM < DTM - 1 m"] = f"{100*(ndsm < -1).mean():.1f}%"
    notes["pixels with DSM < DTM - 5 m"] = f"{100*(ndsm < -5).mean():.1f}%"
    notes["nodata"] = f"{100*dsm_s.mask.mean():.1f}%"

print(f"diagnostics ({box_km[0]:.1f} x {box_km[1]:.1f} km box around the observer)")
for k, v in notes.items():
    print(f"  {k:<32} {v}")

print("\nassertions")
for label, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")

assert all(checks.values()), "validation failed - inspect before continuing"


## Spatial check


In [ ]:
# Render a decimated DSM hillshade for a visual spatial check.
DECIM = 10

with rasterio.open(dsm_vrt) as src:
    
    out_shape = (src.height // DECIM, src.width // DECIM)
    z = src.read(1, out_shape=out_shape, masked=True).astype("float32")
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]

z = np.ma.filled(z, np.nan)

# Illuminate the surface from the northwest.
dy, dx = np.gradient(z, DECIM)
slope = np.arctan(np.hypot(dx, dy))
aspect = np.arctan2(-dx, dy)
az, alt = np.radians(315.0), np.radians(45.0)
hs = (np.sin(alt) * np.cos(slope)
      + np.cos(alt) * np.sin(slope) * np.cos(az - aspect))

fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(hs, cmap="gray", extent=extent, origin="upper")
ax.plot(observers["x_l72"], observers["y_l72"], "o", color="crimson", ms=7)
for _, r in observers.iterrows():
    ax.annotate(r["name"], (r["x_l72"], r["y_l72"]), xytext=(7, 4),
                textcoords="offset points", fontsize=8, color="crimson")

ax.set_xlabel("Lambert 72 X (m)")
ax.set_ylabel("Lambert 72 Y (m)")
ax.set_title(f"DSM hillshade, {DECIM} m — acquired extent")

fig.tight_layout()
fig.savefig(FIGURES / "02_dsm_hillshade.png", dpi=150)
show()
